# 랭체인 멀티턴

- LangChain의 RunnableWithMessageHistory를 사용하여 대화기록을 관리하는 챗봇 구현
- 세션 ID별로 대화 기록이 어떻게 분리되어 저장되는지 확인

## 라이브러리 불러오기

In [1]:
from langchain_core.chat_history import InMemoryChatMessageHistory  # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory  # 메시지 기록을 활용해 실행 가능한 래퍼wrapper 클래스
from langchain_openai import ChatOpenAI  # 오픈AI 모델을 사용하는 랭체인 챗봇 클래스
from langchain_core.messages import HumanMessage

c:\anaconda\envs\LLM\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## API KEY 불러오기

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os

In [3]:
load_dotenv()

True

## 대화 기록 함수 정의

In [4]:
model = ChatOpenAI(model="gpt-4o-mini")

# 세션별 대화 기록을 저장할 딕셔너리
store = {}

# 세션 ID에 따라 대화 기록을 가져오는 함수
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 store에 없으면, 새로 생성해 추가함
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 메모리에 대화 기록을 저장하는 객체 생성
    return store[session_id]  # 해당 세션의 대화 기록을 반환

# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
with_message_history = RunnableWithMessageHistory(model, get_session_history)

## 첫 번째 대화

In [5]:
config = {"configurable": {"session_id": "abc2"}}  # 세션 ID를 설정하는 config 객체 생성

response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 김정은이야.")],
    config=config,
)

print(response.content)

안녕하세요, 김정은님! 어떻게 도와드릴까요?


## 두 번째 대화

In [6]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 김정은입니다. 다른 질문이나 궁금한 점이 있으시면 말씀해 주세요!


## 세션 테스트

In [8]:
config = {"configurable": {"session_id": "abc3"}}

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

response.content

'죄송하지만, 제가 당신의 이름을 알 수 있는 방법이 없습니다. 당신의 이름을 제시해 주시면 그에 대해 이야기할 수 있습니다. 어떻게 도와드릴까요?'

In [9]:
config = {"configurable": {"session_id": "abc2"}}

response = with_message_history.invoke(
    [HumanMessage(content="아까 우리가 무슨 얘기 했지?")],
    config=config,
)

response.content

'우리는 당신의 이름이 김정은이고, 제가 어떻게 도와드릴 수 있는지에 대해 이야기하고 있었습니다. 다른 주제에 대해 이야기하고 싶으시면 말씀해 주세요!'

## 스트리밍 응답 출력

In [10]:
config = {"configurable": {"session_id": "abc2"}}
for r in with_message_history.stream(
    [HumanMessage(content = "내가 어느 나라 사람인지 맞춰보고, 그 나라의 문화에 대해 말해봐")],
    config=config,
):
    print(r.content, end="")
    # print(r.content, end="|")

김정은이라는 이름은 주로 북한에서 사용되는 이름입니다. 북한의 문화는 독특한 역사와 전통을 가지고 있습니다. 예를 들어, 북한에서는 강한 집단주의와 국가주의가 강조됩니다.

하나의 예로, 북한은 매년 김일성 생일과 같은 중요한 국가 기념일을 크게 기념합니다. 또한, 음식 문화에서는 김치, 냉면, 그리고 다양한 국수 요리가 많이 소비됩니다. 북한의 전통 공연 예술도 상당히 발달해 있으며, 군대 행진과 같은 대중 공연이 자주 열립니다.

더 알고 싶은 구체적인 부분이 있으면 말씀해 주세요!